# End-to-End Pipeline Evaluation
## Stage 1 (LOS/NLOS) -> Stage 2 (Correctable?) -> Stage 3 (Error Correction)

**Purpose**: Compare all 4 encoder architectures (LNN, LSTM, CNN, BERT) in a realistic chained pipeline on unseen data.

No ground-truth labels are used for filtering — each stage uses the previous stage's predictions.

Ground truth is only used for **evaluation** (measuring accuracy of the pipeline output).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## Section 1: Configuration

In [ ]:
CONFIG = {
    "search_start": 740,
    "search_end": 810,
}

# Model directories (relative to this notebook)
MODEL_DIRS = {
    "LNN":  "lnn",
    "LSTM": "lstm",
    "CNN":  "cnn",
    "BERT": "bert",
}

UNSEEN_PATH = "dataset/channels/unseen_dataset.csv"

---
## Section 2: Model Definitions

In [ ]:
# ==========================================
# LNN: DualCircuit_PI_HLNN_NoFP
# ==========================================
class PILiquidCell(nn.Module):
    def __init__(self, input_size, hidden_size, ode_unfolds=6):
        super().__init__()
        self.hidden_size = hidden_size
        self.input_size  = input_size
        self.ode_unfolds = ode_unfolds
        self.gleak = nn.Parameter(torch.empty(hidden_size).uniform_(0.001, 1.0))
        self.vleak = nn.Parameter(torch.empty(hidden_size).uniform_(-0.2, 0.2))
        self.cm    = nn.Parameter(torch.empty(hidden_size).uniform_(0.4, 0.6))
        self.w     = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(0.001, 1.0))
        self.erev  = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(-0.2, 0.2))
        self.mu    = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(0.3, 0.8))
        self.sigma = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(3, 8))
        self.sensory_w     = nn.Parameter(torch.empty(input_size, hidden_size).uniform_(0.001, 1.0))
        self.sensory_mu    = nn.Parameter(torch.empty(input_size, hidden_size).uniform_(0.3, 0.8))
        self.sensory_sigma = nn.Parameter(torch.empty(input_size, hidden_size).uniform_(3, 8))

    def forward(self, x_t, h_prev, dt=1.0):
        gleak     = F.softplus(self.gleak)
        cm        = F.softplus(self.cm)
        w         = F.softplus(self.w)
        sensory_w = F.softplus(self.sensory_w)
        sensory_gate    = torch.sigmoid(self.sensory_sigma * (x_t.unsqueeze(-1) - self.sensory_mu))
        sensory_current = (sensory_w * sensory_gate * x_t.unsqueeze(-1)).sum(dim=1)
        cm_t = cm / (dt / self.ode_unfolds)
        v    = h_prev
        for _ in range(self.ode_unfolds):
            recurrent_gate = torch.sigmoid(self.sigma.unsqueeze(0) * (v.unsqueeze(2) - self.mu.unsqueeze(0)))
            w_gate = w.unsqueeze(0) * recurrent_gate
            w_num  = (w_gate * self.erev.unsqueeze(0)).sum(dim=1)
            w_den  = w_gate.sum(dim=1)
            numerator   = cm_t * v + gleak * self.vleak + w_num + sensory_current
            denominator = cm_t + gleak + w_den + 1e-8
            v = numerator / denominator
            v = torch.clamp(v, -1.0, 1.0)
        tau = cm / (gleak + w_den + 1e-8)
        return v, tau


class DualCircuit_PI_HLNN_NoFP(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, dropout=0.2, ode_unfolds=6):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell_los  = PILiquidCell(input_size, hidden_size, ode_unfolds)
        self.cell_nlos = PILiquidCell(input_size, hidden_size, ode_unfolds)
        self.P_nlos2los = nn.Linear(hidden_size, hidden_size, bias=False)
        self.P_los2nlos = nn.Linear(hidden_size, hidden_size, bias=False)
        self.gate_los  = nn.Linear(hidden_size * 2, hidden_size)
        self.gate_nlos = nn.Linear(hidden_size * 2, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(hidden_size, 1), nn.Sigmoid())

    def _run_circuits(self, x_seq):
        batch_size, seq_len, _ = x_seq.size()
        h_los  = torch.zeros(batch_size, self.hidden_size, device=x_seq.device)
        h_nlos = torch.zeros(batch_size, self.hidden_size, device=x_seq.device)
        los_states, nlos_states = [], []
        tau_los_sum  = torch.zeros_like(h_los)
        tau_nlos_sum = torch.zeros_like(h_nlos)
        for t in range(seq_len):
            x_t = x_seq[:, t, :]
            proj_nlos_to_los = self.P_nlos2los(h_nlos)
            proj_los_to_nlos = self.P_los2nlos(h_los)
            g_los  = torch.sigmoid(self.gate_los( torch.cat([h_los,  proj_nlos_to_los], dim=1)))
            g_nlos = torch.sigmoid(self.gate_nlos(torch.cat([h_nlos, proj_los_to_nlos], dim=1)))
            h_los_in  = h_los  + g_los  * proj_nlos_to_los
            h_nlos_in = h_nlos + g_nlos * proj_los_to_nlos
            h_los,  tau_los  = self.cell_los( x_t, h_los_in)
            h_nlos, tau_nlos = self.cell_nlos(x_t, h_nlos_in)
            los_states.append(h_los.unsqueeze(1))
            nlos_states.append(h_nlos.unsqueeze(1))
            tau_los_sum  += tau_los
            tau_nlos_sum += tau_nlos
        los_all  = torch.cat(los_states,  dim=1)
        nlos_all = torch.cat(nlos_states, dim=1)
        tau_los_mean  = tau_los_sum  / seq_len
        tau_nlos_mean = tau_nlos_sum / seq_len
        return los_all, nlos_all, tau_los_mean, tau_nlos_mean

    def _pool_and_fuse(self, los_all, nlos_all):
        return torch.cat([los_all.mean(dim=1), nlos_all.mean(dim=1)], dim=1)

    def forward(self, x_seq):
        los_all, nlos_all, tau_los_mean, tau_nlos_mean = self._run_circuits(x_seq)
        h_fused = self._pool_and_fuse(los_all, nlos_all)
        pred = self.classifier(h_fused)
        return pred, tau_los_mean, tau_nlos_mean

    def embed(self, x_seq):
        los_all, nlos_all, _, _ = self._run_circuits(x_seq)
        return self._pool_and_fuse(los_all, nlos_all)


# ==========================================
# LSTM
# ==========================================
class LSTM_Classifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                           num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x_seq, return_dynamics=False):
        batch_size = x_seq.size(0)
        h0 = torch.zeros(1, batch_size, self.hidden_size, device=x_seq.device)
        c0 = torch.zeros(1, batch_size, self.hidden_size, device=x_seq.device)
        h_all, (h_n, c_n) = self.lstm(x_seq, (h0, c0))
        h_avg = h_all.mean(dim=1)
        pred = self.classifier(h_avg)
        if return_dynamics:
            return pred, h_all
        return pred

    def embed(self, x_seq):
        batch_size = x_seq.size(0)
        h0 = torch.zeros(1, batch_size, self.hidden_size, device=x_seq.device)
        c0 = torch.zeros(1, batch_size, self.hidden_size, device=x_seq.device)
        h_all, _ = self.lstm(x_seq, (h0, c0))
        return h_all.mean(dim=1)


# ==========================================
# CNN
# ==========================================
class CNN_Classifier(nn.Module):
    def __init__(self, input_channels=1, embedding_size=128, dropout=0.4):
        super().__init__()
        self.embedding_size = embedding_size
        self.encoder = nn.Sequential(
            nn.Conv1d(input_channels, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, padding=2, stride=2),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, embedding_size, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm1d(embedding_size), nn.ReLU())
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_size, 32), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(32, 1), nn.Sigmoid())

    def _encode_cir(self, x):
        return self.gap(self.encoder(x)).squeeze(-1)

    def forward(self, x, return_dynamics=False):
        cnn_embed = self._encode_cir(x)
        pred = self.classifier(cnn_embed)
        if return_dynamics:
            return pred, self.encoder(x)
        return pred

    def embed(self, x):
        return self._encode_cir(x)


# ==========================================
# BERT (Transformer)
# ==========================================
class BERT_Classifier(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.2, max_seq_len=60):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1), nn.Sigmoid())

    def _encode(self, x_seq):
        x = self.input_proj(x_seq)
        x = x + self.pos_embed[:, :x.size(1), :]
        return self.transformer(x)

    def forward(self, x_seq, return_dynamics=False):
        h_all = self._encode(x_seq)
        h_avg = self.norm(h_all.mean(dim=1))
        pred = self.classifier(h_avg)
        if return_dynamics:
            return pred, h_all
        return pred

    def embed(self, x_seq):
        h_all = self._encode(x_seq)
        return self.norm(h_all.mean(dim=1))


print("All model classes defined.")

---
## Section 3: Load Unseen Data

In [ ]:
# ==========================================
# HELPER: ROI alignment (for CIR preprocessing)
# ==========================================
def get_roi_alignment(sig, search_start=CONFIG["search_start"],
                      search_end=CONFIG["search_end"]):
    """Find leading edge by backtracking from peak."""
    region = sig[search_start:search_end]
    if len(region) == 0:
        return np.argmax(sig)
    peak_local = np.argmax(region)
    peak_idx = search_start + peak_local
    peak_val = sig[peak_idx]
    noise_section = sig[:search_start]
    if len(noise_section) > 10:
        noise_mean = np.mean(noise_section)
        noise_std = np.std(noise_section)
        threshold = max(noise_mean + 3 * noise_std, 0.05 * peak_val)
    else:
        threshold = 0.05 * peak_val
    leading_edge = peak_idx
    for i in range(peak_idx, max(search_start - 20, 0), -1):
        if sig[i] < threshold:
            leading_edge = i + 1
            break
    return leading_edge


def preprocess_cir(sig, leading_edge, pre_crop, total_len):
    """Convert RXPACC-normalized CIR to windowed, min-max normalized sequence."""
    start = max(0, leading_edge - pre_crop)
    end = start + total_len
    if end > len(sig):
        end = len(sig)
        start = max(0, end - total_len)
    crop = sig[start:end]
    if len(crop) < total_len:
        crop = np.pad(crop, (0, total_len - len(crop)), mode='constant')
    local_min, local_max = np.min(crop), np.max(crop)
    rng = local_max - local_min
    crop = (crop - local_min) / rng if rng > 0 else np.zeros(total_len)
    return crop


# ==========================================
# LOAD ALL UNSEEN DATA
# ==========================================
print(f"Loading unseen data: {UNSEEN_PATH}")
df_unseen = pd.read_csv(UNSEEN_PATH)
cir_cols = sorted([c for c in df_unseen.columns if c.startswith('CIR')],
                  key=lambda x: int(x.replace('CIR', '')))

all_sigs, all_les = [], []
all_d_hw, all_d_direct, all_labels, all_source = [], [], [], []

for idx, row in df_unseen.iterrows():
    sig = pd.to_numeric(row[cir_cols], errors='coerce').fillna(0).astype(float).values
    rxpacc = float(row.get('RXPACC', 128.0))
    if rxpacc > 0:
        sig = sig / rxpacc
    le = get_roi_alignment(sig)
    all_sigs.append(sig)
    all_les.append(le)
    all_d_hw.append(float(row['Distance']))
    all_d_direct.append(float(row['d_direct']))
    all_labels.append(int(row['Label']))
    all_source.append(str(row.get('Source_File', '')))

all_d_hw = np.array(all_d_hw)
all_d_direct = np.array(all_d_direct)
all_labels = np.array(all_labels)

# Ground truth from pre-labeled CSV columns
gt_nlos = all_labels == 1
gt_correctable = ((df_unseen["Label"] == 1) & (df_unseen["is_correctable"] == 1)).values

# Scenario groups
all_groups = []
for sf in all_source:
    match = re.search(r'([\d.]+)m_(nlos|los)', sf)
    all_groups.append(match.group(1) + 'm_' + match.group(2) if match else 'unknown')
all_groups = np.array(all_groups)

print(f"  Total unseen: {len(df_unseen)}")
print(f"  Ground truth NLOS: {gt_nlos.sum()}")
print(f"  Ground truth correctable: {gt_correctable.sum()}")
print(f"  LOS: {(~gt_nlos).sum()}")

---
## Section 4: Load All Encoders & RF Models

In [ ]:
# ==========================================
# LOAD ALL 4 ENCODERS + STAGE 2/3 RF MODELS
# ==========================================
models = {}

# ── LNN ──
d = MODEL_DIRS["LNN"]
_saved = torch.load(f"{d}/stage1_pi_hlnn_no_fp_v3_config.pt", map_location="cpu", weights_only=False)
lnn_config = _saved["config"]
lnn_encoder = DualCircuit_PI_HLNN_NoFP(
    input_size=lnn_config['input_size'], hidden_size=lnn_config['hidden_size'],
    dropout=lnn_config['dropout'], ode_unfolds=lnn_config['ode_unfolds']).to(device)
lnn_encoder.load_state_dict(torch.load(f"{d}/stage1_pi_hlnn_no_fp_v3_best.pt", map_location=device, weights_only=True))
lnn_encoder.eval()
for p in lnn_encoder.parameters(): p.requires_grad = False
models["LNN"] = {
    "encoder": lnn_encoder, "config": lnn_config,
    "stage2_rf": joblib.load(f"{d}/stage2_bounce_rf.joblib"),
    "stage3_rf": joblib.load(f"{d}/stage3_nlos_bias_rf.joblib"),
    "forward_fn": lambda enc, batch: enc(batch)[0],  # LNN returns (pred, tau_los, tau_nlos)
    "tensor_reshape": lambda seqs, total_len: seqs.reshape(-1, total_len, 1),
}
print(f"LNN loaded: emb_dim={lnn_config['hidden_size']*2}, params={sum(p.numel() for p in lnn_encoder.parameters()):,}")

# ── LSTM ──
d = MODEL_DIRS["LSTM"]
_saved = torch.load(f"{d}/stage1_lstm_config.pt", map_location="cpu", weights_only=False)
lstm_config = _saved["config"]
lstm_encoder = LSTM_Classifier(
    input_size=lstm_config['input_size'], hidden_size=lstm_config['hidden_size'],
    dropout=lstm_config['dropout']).to(device)
lstm_encoder.load_state_dict(torch.load(f"{d}/stage1_lstm_best.pt", map_location=device, weights_only=True))
lstm_encoder.eval()
for p in lstm_encoder.parameters(): p.requires_grad = False
models["LSTM"] = {
    "encoder": lstm_encoder, "config": lstm_config,
    "stage2_rf": joblib.load(f"{d}/stage2_bounce_rf.joblib"),
    "stage3_rf": joblib.load(f"{d}/stage3_nlos_bias_rf.joblib"),
    "forward_fn": lambda enc, batch: enc(batch),
    "tensor_reshape": lambda seqs, total_len: seqs.reshape(-1, total_len, 1),
}
print(f"LSTM loaded: emb_dim={lstm_config['hidden_size']}, params={sum(p.numel() for p in lstm_encoder.parameters()):,}")

# ── CNN ──
d = MODEL_DIRS["CNN"]
_saved = torch.load(f"{d}/stage1_cnn_config.pt", map_location="cpu", weights_only=False)
cnn_config = _saved["config"]
cnn_encoder = CNN_Classifier(
    input_channels=cnn_config['input_channels'],
    embedding_size=cnn_config['embedding_size'],
    dropout=cnn_config['dropout']).to(device)
cnn_encoder.load_state_dict(torch.load(f"{d}/stage1_cnn_best.pt", map_location=device, weights_only=True))
cnn_encoder.eval()
for p in cnn_encoder.parameters(): p.requires_grad = False
models["CNN"] = {
    "encoder": cnn_encoder, "config": cnn_config,
    "stage2_rf": joblib.load(f"{d}/stage2_bounce_rf.joblib"),
    "stage3_rf": joblib.load(f"{d}/stage3_nlos_bias_rf.joblib"),
    "forward_fn": lambda enc, batch: enc(batch),
    "tensor_reshape": lambda seqs, total_len: seqs.reshape(-1, 1, total_len),  # channels-first
}
print(f"CNN loaded: emb_dim={cnn_config['embedding_size']}, params={sum(p.numel() for p in cnn_encoder.parameters()):,}")

# ── BERT ──
d = MODEL_DIRS["BERT"]
_saved = torch.load(f"{d}/stage1_bert_config.pt", map_location="cpu", weights_only=False)
bert_config = _saved["config"]
bert_encoder = BERT_Classifier(
    input_size=bert_config['input_size'], d_model=bert_config['d_model'],
    nhead=bert_config['nhead'], num_layers=bert_config['num_layers'],
    dim_feedforward=bert_config['dim_feedforward'],
    dropout=bert_config['dropout'], max_seq_len=bert_config['total_len']).to(device)
bert_encoder.load_state_dict(torch.load(f"{d}/stage1_bert_best.pt", map_location=device, weights_only=True))
bert_encoder.eval()
for p in bert_encoder.parameters(): p.requires_grad = False
models["BERT"] = {
    "encoder": bert_encoder, "config": bert_config,
    "stage2_rf": joblib.load(f"{d}/stage2_bounce_rf.joblib"),
    "stage3_rf": joblib.load(f"{d}/stage3_nlos_bias_rf.joblib"),
    "forward_fn": lambda enc, batch: enc(batch),
    "tensor_reshape": lambda seqs, total_len: seqs.reshape(-1, total_len, 1),
}
print(f"BERT loaded: emb_dim={bert_config['d_model']}, params={sum(p.numel() for p in bert_encoder.parameters()):,}")

print(f"\nAll 4 models loaded successfully.")

---
## Section 5: Run Chained Pipeline for Each Model

In [ ]:
# ==========================================
# CHAINED PIPELINE: Stage 1 -> Stage 2 -> Stage 3
# ==========================================
results = {}

for name, m in models.items():
    print(f"\n{'='*60}")
    print(f"Running pipeline: {name}")
    print(f"{'='*60}")

    encoder = m["encoder"]
    cfg = m["config"]
    forward_fn = m["forward_fn"]
    reshape_fn = m["tensor_reshape"]
    stage2_rf = m["stage2_rf"]
    stage3_rf = m["stage3_rf"]

    pre_crop = cfg.get('pre_crop', 10)
    total_len = cfg.get('total_len', 60)

    # Preprocess all CIR
    cir_sequences = []
    for i in range(len(all_sigs)):
        crop = preprocess_cir(all_sigs[i], all_les[i], pre_crop, total_len)
        cir_sequences.append(crop)

    cir_tensor = torch.tensor(
        reshape_fn(np.array(cir_sequences), total_len),
        dtype=torch.float32
    ).to(device)

    # ── Stage 1: Predict NLOS ──
    stage1_probs = []
    with torch.no_grad():
        for i in range(0, len(cir_tensor), 256):
            batch = cir_tensor[i:i+256]
            pred = forward_fn(encoder, batch)
            stage1_probs.append(pred.cpu().numpy())
    stage1_probs = np.concatenate(stage1_probs).flatten()
    predicted_nlos = stage1_probs >= 0.5
    predicted_nlos_idx = np.where(predicted_nlos)[0]

    s1_acc = (predicted_nlos == gt_nlos).mean()
    print(f"  Stage 1: {predicted_nlos.sum()}/{len(df_unseen)} predicted NLOS (acc={s1_acc:.4f})")

    # ── Stage 2: Predict Correctable ──
    nlos_embeddings = []
    with torch.no_grad():
        for i in range(0, len(predicted_nlos_idx), 256):
            batch_idx = predicted_nlos_idx[i:i+256]
            batch = cir_tensor[batch_idx]
            emb = encoder.embed(batch)
            nlos_embeddings.append(emb.cpu().numpy())
    nlos_embeddings = np.vstack(nlos_embeddings)

    stage2_preds = stage2_rf.predict(nlos_embeddings)
    predicted_correctable_mask = stage2_preds == 1
    correctable_idx = predicted_nlos_idx[predicted_correctable_mask]

    print(f"  Stage 2: {predicted_correctable_mask.sum()}/{len(predicted_nlos_idx)} predicted correctable")

    # ── Stage 3: Predict Ranging Error ──
    corr_embeddings = []
    with torch.no_grad():
        for i in range(0, len(correctable_idx), 256):
            batch_idx = correctable_idx[i:i+256]
            batch = cir_tensor[batch_idx]
            emb = encoder.embed(batch)
            corr_embeddings.append(emb.cpu().numpy())
    corr_embeddings = np.vstack(corr_embeddings)

    predicted_errors = stage3_rf.predict(corr_embeddings)

    # ── Evaluate ──
    d_hw_pipe = all_d_hw[correctable_idx]
    d_dir_pipe = all_d_direct[correctable_idx]
    labels_pipe = all_labels[correctable_idx]
    gt_corr_pipe = gt_correctable[correctable_idx]
    groups_pipe = all_groups[correctable_idx]

    raw_errors_pipe = d_hw_pipe - d_dir_pipe
    corrected_distances_pipe = d_hw_pipe - predicted_errors
    corrected_errors_pipe = corrected_distances_pipe - d_dir_pipe

    actually_nlos_mask = labels_pipe == 1

    print(f"  Pipeline: {len(df_unseen)} -> {predicted_nlos.sum()} -> {len(correctable_idx)}")
    print(f"    Actually NLOS: {actually_nlos_mask.sum()}, LOS FP: {(~actually_nlos_mask).sum()}")
    print(f"    Actually correctable (GT): {gt_corr_pipe.sum()}")

    # NLOS correction metrics
    if actually_nlos_mask.sum() > 0:
        raw_mae = np.abs(raw_errors_pipe[actually_nlos_mask]).mean()
        corr_mae = np.abs(corrected_errors_pipe[actually_nlos_mask]).mean()
        print(f"    NLOS Before: MAE={raw_mae:.4f}m")
        print(f"    NLOS After:  MAE={corr_mae:.4f}m")
        if corr_mae > 0:
            print(f"    Improvement: {raw_mae/corr_mae:.1f}x")

    # Overall metrics
    overall_raw_mae = np.abs(raw_errors_pipe).mean()
    overall_corr_mae = np.abs(corrected_errors_pipe).mean()

    # Store results
    results[name] = {
        "s1_predicted_nlos": int(predicted_nlos.sum()),
        "s1_accuracy": float(s1_acc),
        "s2_predicted_correctable": len(correctable_idx),
        "actually_nlos": int(actually_nlos_mask.sum()),
        "los_fp": int((~actually_nlos_mask).sum()),
        "gt_correctable_in_pipe": int(gt_corr_pipe.sum()),
        "nlos_raw_mae": float(np.abs(raw_errors_pipe[actually_nlos_mask]).mean()) if actually_nlos_mask.sum() > 0 else None,
        "nlos_corr_mae": float(np.abs(corrected_errors_pipe[actually_nlos_mask]).mean()) if actually_nlos_mask.sum() > 0 else None,
        "overall_raw_mae": float(overall_raw_mae),
        "overall_corr_mae": float(overall_corr_mae),
        "correctable_idx": correctable_idx,
        "predicted_errors": predicted_errors,
        "raw_errors_pipe": raw_errors_pipe,
        "corrected_errors_pipe": corrected_errors_pipe,
        "groups_pipe": groups_pipe,
        "actually_nlos_mask": actually_nlos_mask,
        "predicted_nlos": predicted_nlos,
    }

---
## Section 6: Side-by-Side Comparison

In [ ]:
# ==========================================
# COMPARISON TABLE
# ==========================================
print(f"{'='*80}")
print(f"CHAINED PIPELINE COMPARISON — Unseen Data ({len(df_unseen)} samples)")
print(f"{'='*80}")

header = f"{'Model':<8} {'S1 NLOS':>8} {'S1 Acc':>8} {'S2 Corr':>8} {'NLOS':>6} {'LOS FP':>7} {'GT Corr':>8} {'Raw MAE':>9} {'Corr MAE':>9} {'Improv':>7}"
print(header)
print('-' * len(header))

for name in ["LNN", "LSTM", "CNN", "BERT"]:
    r = results[name]
    improvement = f"{r['nlos_raw_mae']/r['nlos_corr_mae']:.1f}x" if r['nlos_corr_mae'] and r['nlos_corr_mae'] > 0 else 'N/A'
    print(f"{name:<8} {r['s1_predicted_nlos']:>8} {r['s1_accuracy']:>8.4f} {r['s2_predicted_correctable']:>8} "
          f"{r['actually_nlos']:>6} {r['los_fp']:>7} {r['gt_correctable_in_pipe']:>8} "
          f"{r['nlos_raw_mae']:>9.4f} {r['nlos_corr_mae']:>9.4f} {improvement:>7}")

In [ ]:
# ==========================================
# VISUALIZATION: Pipeline Funnel Comparison
# ==========================================
fig, axs = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

for ax, name in zip(axs, ["LNN", "LSTM", "CNN", "BERT"]):
    r = results[name]
    stages = ['All\nUnseen', 'S1\nNLOS', 'S2\nCorr.']
    counts = [len(df_unseen), r['s1_predicted_nlos'], r['s2_predicted_correctable']]
    colors_bar = ['#3498db', '#e67e22', '#2ecc71']
    bars = ax.bar(stages, counts, color=colors_bar, edgecolor='white', alpha=0.8)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(c), ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

axs[0].set_ylabel('Sample Count')
plt.suptitle('Pipeline Funnel: Stage 1 -> Stage 2 -> Stage 3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# VISUALIZATION: Before vs After MAE Comparison
# ==========================================
fig, ax = plt.subplots(figsize=(10, 6))

model_names = ["LNN", "LSTM", "CNN", "BERT"]
raw_maes = [results[n]['nlos_raw_mae'] for n in model_names]
corr_maes = [results[n]['nlos_corr_mae'] for n in model_names]

x_pos = np.arange(len(model_names))
w = 0.35
bars1 = ax.bar(x_pos - w/2, raw_maes, w, color='#e74c3c', alpha=0.7, label='Before ML')
bars2 = ax.bar(x_pos + w/2, corr_maes, w, color='#2ecc71', alpha=0.7, label='After ML')

for i in range(len(x_pos)):
    if raw_maes[i] and corr_maes[i] and corr_maes[i] > 0:
        imp = raw_maes[i] / corr_maes[i]
        ax.text(x_pos[i], max(raw_maes[i], corr_maes[i]) + 0.05,
                f'{imp:.1f}x', ha='center', fontsize=11, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(model_names, fontsize=12)
ax.set_ylabel('MAE (m)', fontsize=12)
ax.set_title('NLOS Distance Correction: Before vs After (Pipeline NLOS Samples)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# VISUALIZATION: Per-Scenario Breakdown (all models)
# ==========================================
# Collect all NLOS scenarios across models
all_nlos_scenarios = set()
for name in model_names:
    r = results[name]
    nlos_groups = r['groups_pipe'][r['actually_nlos_mask']]
    all_nlos_scenarios.update(nlos_groups)
all_nlos_scenarios = sorted(all_nlos_scenarios)

fig, axs = plt.subplots(1, len(all_nlos_scenarios), figsize=(5*len(all_nlos_scenarios), 5), sharey=True)
if len(all_nlos_scenarios) == 1:
    axs = [axs]

for ax, scenario in zip(axs, all_nlos_scenarios):
    before_vals, after_vals = [], []
    valid_models = []
    for name in model_names:
        r = results[name]
        mask = (r['groups_pipe'] == scenario) & r['actually_nlos_mask']
        if mask.sum() > 0:
            before_vals.append(np.abs(r['raw_errors_pipe'][mask]).mean())
            after_vals.append(np.abs(r['corrected_errors_pipe'][mask]).mean())
            valid_models.append(name)

    if valid_models:
        x = np.arange(len(valid_models))
        w = 0.35
        ax.bar(x - w/2, before_vals, w, color='#e74c3c', alpha=0.7, label='Before')
        ax.bar(x + w/2, after_vals, w, color='#2ecc71', alpha=0.7, label='After')
        ax.set_xticks(x)
        ax.set_xticklabels(valid_models)
        ax.legend(fontsize=8)
        # Count from first valid model
        r0 = results[valid_models[0]]
        n = ((r0['groups_pipe'] == scenario) & r0['actually_nlos_mask']).sum()
    ax.set_title(f'{scenario}', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

axs[0].set_ylabel('MAE (m)')
plt.suptitle('Per-Scenario NLOS Correction by Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# VISUALIZATION: Predicted vs Actual Ranging Error (all models)
# ==========================================
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

for ax, name in zip(axs.flat, model_names):
    r = results[name]
    mask = r['actually_nlos_mask']
    if mask.sum() > 0:
        actual_re = r['raw_errors_pipe'][mask]
        pred_re = r['predicted_errors'][mask]
        for g in sorted(set(r['groups_pipe'][mask])):
            g_mask = (r['groups_pipe'] == g) & mask
            if g_mask.sum() > 0:
                ax.scatter(r['raw_errors_pipe'][g_mask], r['predicted_errors'][g_mask],
                          s=20, alpha=0.5, label=g)
        lims = [min(actual_re.min(), pred_re.min()) - 0.2,
                max(actual_re.max(), pred_re.max()) + 0.2]
        ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.5)
        r2 = r2_score(actual_re, pred_re) if len(actual_re) > 1 else float('nan')
        mae = np.abs(actual_re - pred_re).mean()
        ax.set_title(f'{name} (R2={r2:.4f}, MAE={mae:.3f}m)', fontweight='bold')
    else:
        ax.set_title(f'{name} (no NLOS samples)')
    ax.set_xlabel('Actual RE (m)')
    ax.set_ylabel('Predicted RE (m)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual Ranging Error (Pipeline NLOS)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 7: Summary

In [ ]:
print("=" * 70)
print("END-TO-END PIPELINE SUMMARY")
print("=" * 70)
print(f"Unseen dataset: {len(df_unseen)} samples ({gt_nlos.sum()} NLOS, {(~gt_nlos).sum()} LOS)")
print(f"Ground truth correctable: {gt_correctable.sum()}")
print()

# Find best model
best_model = min(model_names, key=lambda n: results[n]['nlos_corr_mae'] or float('inf'))
print(f"Best pipeline (lowest NLOS corrected MAE): {best_model}")
print(f"  MAE = {results[best_model]['nlos_corr_mae']:.4f}m")
print(f"  Improvement = {results[best_model]['nlos_raw_mae']/results[best_model]['nlos_corr_mae']:.1f}x")
print()

for name in model_names:
    r = results[name]
    imp = f"{r['nlos_raw_mae']/r['nlos_corr_mae']:.1f}x" if r['nlos_corr_mae'] and r['nlos_corr_mae'] > 0 else 'N/A'
    print(f"  {name}: S1={r['s1_predicted_nlos']}, S2={r['s2_predicted_correctable']}, "
          f"NLOS MAE {r['nlos_raw_mae']:.4f} -> {r['nlos_corr_mae']:.4f}m ({imp}), "
          f"LOS FP={r['los_fp']}")